In [1]:
#*************************************************************************
#   > Filename    : make_gc_bit_great_again.py
#   > Description : GIN trained on PROTEINS
#*************************************************************************


## Setup

## Packages and Libraries

In [1]:
import os
import numpy as np
import random
import torch
import torch.nn as nn
from torch.nn import Linear, Sequential, ReLU, Identity, BatchNorm1d as BN
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import add_self_loops, degree,remove_self_loops
from torch_geometric.data import Data
from torch_geometric.datasets import TUDataset,Planetoid,GNNBenchmarkDataset
from torch_geometric.loader import DataLoader
from torch_scatter import scatter_mean
from torch.autograd.function import InplaceFunction
from torch_geometric.nn import GCNConv,GINConv,global_mean_pool,TopKPooling
import torch_geometric.transforms as T

from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm
import time
import argparse
import statistics as stat
from tabulate import tabulate
#################################################
from quantize_function.u_quant_gc_bit_debug import *
from quantize_function.MessagePassing_gc_bit import GINConvMultiQuant
from quantize_function.get_scale_index import get_deg_index, get_scale_index

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
import itertools
import os

os.environ["DGLBACKEND"] = "pytorch"

import copy
import dgl
import dgl.data
import numpy as np
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.utils.prune as prune
from sklearn.metrics import roc_auc_score
from torchprofile import profile_macs


def get_model_macs(model, inputs) -> int:
    '''
    MACs are a common metric to measure the computational complexity 
    of deep neural networks, as they reflect the number of arithmetic
    operations involved in the forward pass of the model
    '''
    return profile_macs(model, inputs)


def get_sparsity(tensor: torch.Tensor) -> float:
    """
    calculate the sparsity of the given tensor
        sparsity = #zeros / #elements = 1 - #nonzeros / #elements
    """
    return 1 - float(tensor.count_nonzero()) / tensor.numel()


def get_model_sparsity(model: nn.Module) -> float:
    """
    calculate the sparsity of the given model
        sparsity = #zeros / #elements = 1 - #nonzeros / #elements
    """
    num_nonzeros, num_elements = 0, 0
    for param in model.parameters():
        num_nonzeros += param.count_nonzero()
        num_elements += param.numel()
    return 1 - float(num_nonzeros) / num_elements

def get_num_parameters(model: nn.Module, count_nonzero_only=False) -> int:
    """
    calculate the total number of parameters of model
    :param count_nonzero_only: only count nonzero weights
    """
    num_counted_elements = 0
    for param in model.parameters():
        if count_nonzero_only:
            num_counted_elements += param.count_nonzero()
        else:
            num_counted_elements += param.numel()
    return num_counted_elements


def get_model_size(model: nn.Module, data_width=32, count_nonzero_only=False) -> int:
    """
    calculate the model size in bits
    :param data_width: #bits per element
    :param count_nonzero_only: only count nonzero weights
    """
    return get_num_parameters(model, count_nonzero_only) * data_width

Byte = 8
KiB = 1024 * Byte
MiB = 1024 * KiB
GiB = 1024 * MiB



## qGIN Model with Quatization

# Helpful Function

In [4]:
# Relu and Batch Normalization
class relu(nn.Module):
    def __init__(self,):
        super().__init__()
    def forward(self,x,edge_index,bit_sum):
        x[x<0] = 0
        return x,edge_index,bit_sum

class bn(nn.Module):
    def __init__(self,hidden_units):
        super().__init__()
        self.bn = nn.BatchNorm1d(hidden_units)
    def forward(self,x,edge_index,bit_sum):
        x = self.bn(x)
        return x,edge_index,bit_sum

In [5]:
class NormalizedDegree(object):
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def __call__(self, data):
        deg = degree(data.edge_index[0], dtype=torch.float)
        deg = (deg - self.mean) / self.std
        data.x = deg.view(-1, 1)
        return data


def num_graphs(data):
    if data.batch is not None:
        return data.num_graphs
    else:
        return data.x.size(0)


def train(model, optimizer, loader,a_loss, a_storage=1):
    model.train()

    total_loss = 0
    for data in loader:
        optimizer.zero_grad()
        data = data.to(device)      
        out= model(data)
        # out,bit_sum = model(data)
        loss = F.cross_entropy(out, data.y.view(-1))
        #loss_store = a_loss*F.relu(bit_sum-a_storage)**2
        #loss_store.backward(retain_graph=True)
        loss.backward()
        total_loss += loss.item() * num_graphs(data)
        optimizer.step()
    return total_loss / len(loader.dataset)


def eval_acc(model, loader):
    model.eval()

    correct = 0
    for data in loader:
        data = data.to(device)
        with torch.no_grad():
            pred = model(data).max(1)[1]
            #   pred = model(data)[0].max(1)[1]
        correct += pred.eq(data.y.view(-1)).sum().item()
    return correct / len(loader.dataset)


def eval_loss(model, loader):
    model.eval()

    loss = 0
    for data in loader:
        data = data.to(device)
        with torch.no_grad():
            out = model(data)
            #out = model(data)[0]
        loss += F.cross_entropy(out, data.y.view(-1), reduction="sum").item()
    return loss / len(loader.dataset)

def k_fold(dataset, folds):
    skf = StratifiedKFold(folds, shuffle=True, random_state=12345)

    test_indices, train_indices = [], []
    for _, idx in skf.split(torch.zeros(len(dataset)), dataset.data.y):
        test_indices.append(torch.from_numpy(idx))

    val_indices = [test_indices[i - 1] for i in range(folds)]

    for i in range(folds):
        train_mask = torch.ones(len(dataset), dtype=torch.bool)
        train_mask[test_indices[i]] = 0
        train_mask[val_indices[i]] = 0
        train_indices.append(train_mask.nonzero().view(-1))

    return train_indices, test_indices, val_indices



def load_checkpoint(model, checkpoint):
    if checkpoint != 'No':
        print("loading checkpoint...")
        model_dict = model.state_dict()
        modelCheckpoint = torch.load(checkpoint)
        pretrained_dict = modelCheckpoint['state_dict']
        new_dict = {k: v for k, v in pretrained_dict.items() if ((k in model_dict.keys()))}
        model_dict.update(new_dict)
        print('Total : {}, update: {}'.format(len(pretrained_dict), len(new_dict)))
        model.load_state_dict(model_dict)
        print("loaded finished!")
    return model


## Definition of Requirment Parameters as args

In [6]:
import sys
import argparse

# Clearing the arguments
sys.argv = ['']


parser = argparse.ArgumentParser()
parser.add_argument('--model',type=str,default='GIN')
parser.add_argument('--gpu_id',type=int,default=0)
parser.add_argument('--dataset_name',type=str,default='PROTEINS')
parser.add_argument('--num_deg',type=int,default=1000)
parser.add_argument('--num_layers', type=int, default=5)
parser.add_argument('--hidden_units',type=int,default=64)
parser.add_argument('--batch-size',type=int,default=128)
parser.add_argument('--bit',type=int,default=4)
parser.add_argument('--max_epoch',type=int,default=100)
parser.add_argument('--max_cycle',type=int,default=2000)
parser.add_argument('--folds',type=int,default=5)
parser.add_argument('--weight_decay',type=float,default=0)
parser.add_argument('--lr',type=float,default=0.01)
parser.add_argument('--a_loss',type=float,default=0.001)
parser.add_argument('--lr_quant_scale_fea',type=float,default=0.02)
parser.add_argument('--lr_quant_scale_xw',type=float,default=1e-2)
parser.add_argument('--lr_quant_scale_weight',type=float,default=0.02)
parser.add_argument('--lr_quant_bit_fea',type=float,default=0.008)
parser.add_argument('--lr_quant_bit_weight',type=float,default=0.0001)
parser.add_argument('--lr_step_size',type=int, default=50)
parser.add_argument('--lr_decay_factor',type=float,default=0.5)
parser.add_argument('--lr_schedule_patience',type=int,default=10)
parser.add_argument('--is_naive',type=bool,default=False)
###############################################################
parser.add_argument('--resume',type=bool,default=True)
parser.add_argument('--store_ckpt',type=bool,default=True)
parser.add_argument('--uniform',type=bool,default=True)
parser.add_argument('--use_norm_quant',type=bool,default=True)
###############################################################
# The target memory size of nodes features
parser.add_argument('--a_storage',type=float,default=1)
# Path to results
parser.add_argument('--result_folder',type=str,default='result')
# Path to checkpoint
parser.add_argument('--check_folder',type=str,default='checkpoint')
# Path to dataset
parser.add_argument('--path2dataset',type=str,default='/')

args = parser.parse_args()
print(args)

Namespace(model='GIN', gpu_id=0, dataset_name='PROTEINS', num_deg=1000, num_layers=5, hidden_units=64, batch_size=128, bit=4, max_epoch=100, max_cycle=2000, folds=5, weight_decay=0, lr=0.01, a_loss=0.001, lr_quant_scale_fea=0.02, lr_quant_scale_xw=0.01, lr_quant_scale_weight=0.02, lr_quant_bit_fea=0.008, lr_quant_bit_weight=0.0001, lr_step_size=50, lr_decay_factor=0.5, lr_schedule_patience=10, is_naive=False, resume=True, store_ckpt=True, uniform=True, use_norm_quant=True, a_storage=1, result_folder='result', check_folder='checkpoint', path2dataset='/')


In [7]:
###############################################################
model = args.model
dataset_name = args.dataset_name
num_layers = args.num_layers
hidden_units=args.hidden_units
bit=args.bit
max_epoch = args.max_epoch
resume = args.resume
path2result = args.result_folder+'/'+args.model+'_'+dataset_name
path2check = args.check_folder+'/'+args.model+'_'+dataset_name
if not os.path.exists(path2result):
    os.makedirs(path2result)
if not os.path.exists(path2check):
    os.makedirs(path2check)
###############################################################


## Loading Dataset and Normalization

In [8]:

if(args.resume==True):
    file_name = path2result+'/'+args.model+'_'+str(hidden_units)+'_'+'_on_'+dataset_name+'_'+str(bit)+'bit-'+str(max_epoch)+'.txt'
    if not os.path.exists(file_name):
            with open(file_name, 'w') as f:
                for key, value in vars(args).items():
                    f.write('%s:%s\n'%(key, value))
if(args.dataset_name=='PROTEINS'):
    dataset = TUDataset(args.path2dataset, args.dataset_name)

if dataset.data.x is None:
    max_degree = 0
    degs = []
    for data in dataset:
        degs += [degree(data.edge_index[0], dtype=torch.long)]
        max_degree = max(max_degree, degs[-1].max().item())

    if max_degree < 1000:
        dataset.transform = T.OneHotDegree(max_degree)
    else:
        deg = torch.cat(degs, dim=0).to(torch.float)
        mean, std = deg.mean().item(), deg.std().item()
        dataset.transform = NormalizedDegree(mean, std)

C:\Users\Dell\AppData\Local\Programs\Python\Python310\lib\site-packages\torch_geometric\data\in_memory_dataset.py:183: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)


In [12]:
import torch
from torch_geometric.nn import GINConv

class GIN(nn.Module):
    def __init__(self, dataset, num_layers, hidden_units, num_deg=1000, init='norm'):
        super(GIN, self).__init__()
        self.conv1 = GINConv(
            nn.Sequential(
                nn.Linear(3, hidden_units),
                nn.ReLU(),
                nn.Linear(hidden_units, hidden_units),
                nn.ReLU(),
                nn.BatchNorm1d(hidden_units),
            ),
            train_eps=True,
        )
        self.convs = torch.nn.ModuleList()
        
        for _ in range(num_layers - 1):
            self.convs.append(
                GINConv(
                    nn.Sequential(
                        nn.Linear(hidden_units, hidden_units),
                        nn.ReLU(),
                        nn.Linear(hidden_units, hidden_units),
                        nn.ReLU(),
                        nn.BatchNorm1d(hidden_units),
                    ),
                    train_eps=True,
                )
            )
        self.bn_list = torch.nn.ModuleList()
        for _ in range(num_layers):
            self.bn_list.append(nn.BatchNorm1d(hidden_units))
        
        self.lin1 = nn.Linear(hidden_units, hidden_units)
        self.lin2 = nn.Linear(hidden_units, dataset.num_classes)

    def reset_parameters(self):
        self.conv1.reset_parameters()
        for conv in self.convs:
            conv.reset_parameters()
        self.lin1.reset_parameters()
        self.lin2.reset_parameters()

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        bit_sum=x.new_zeros(1)
        x= self.conv1(x, edge_index)
        x = self.bn_list[0](x)
        i = 1
        for conv in self.convs:
            x= conv(x, edge_index)
            x = self.bn_list[i](x)
            i += 1
        x = global_mean_pool(x, batch)
        x = self.lin1(x)
        x = F.relu(x)
        x = self.lin2(x)
        return F.log_softmax(x, dim=-1)



## Training Process

In [45]:

# writer = SummaryWriter(log_dir=path2log)
args.max_epoch=100
max_acc = 0.79
for i in range(4):
    accu = []
    accu_max = []


    for fold, (train_idx, test_idx, val_idx) in enumerate(zip(*k_fold(dataset, args.folds))):
        print_max_acc=0
        train_dataset = dataset[train_idx.tolist()]
        test_dataset = dataset[test_idx.tolist()]
        val_dataset = dataset[val_idx.tolist()]
        train_loader = DataLoader(train_dataset, args.batch_size, shuffle=True,drop_last=False)
        val_loader = DataLoader(val_dataset, args.batch_size, shuffle=False,drop_last=False)
        test_loader = DataLoader(test_dataset, args.batch_size, shuffle=False,drop_last=False)
        k=0
        model=GIN(train_dataset, args.num_layers,hidden_units=args.hidden_units,
                num_deg=args.num_deg).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=args.lr_step_size, gamma=args.lr_decay_factor)

        # if (os.path.exists(path2check)):
        #     model = load_checkpoint(model,path2check)

        for epoch in range(args.max_epoch):
            t = tqdm(epoch)
            train_loss=0
            train_loss = train(model,optimizer,train_loader,args.a_loss, args.a_storage)
            start = time.process_time()
            val_loss = eval_loss(model,val_loader)
            end = time.process_time()
            acc = eval_acc(model,test_loader)
            # for name,param in model.named_parameters():
            #     a=param.grad
            #     if(a!=None):
            #         writer.add_histogram(tag=name+'_grad', values=a, global_step=epoch)
            scheduler.step()
            t.set_postfix(
                        {
                            "Train_Loss": "{:05.3f}".format(train_loss),
                            "Val_Loss": "{:05.3f}".format(val_loss),
                            "Acc": "{:05.3f}".format(acc),
                            "Epoch":"{:05.1f}".format(epoch),
                            "Fold":"{:05.1f}".format(fold),
                        }
                    )
            accu.append(acc)
            if(acc>max_acc):
                path = path2check+'/'+args.model+'_'+str(hidden_units)+'_on_'+dataset_name+'_'+str(bit)+'bit-'+str(max_epoch)+'.pth.tar'
                max_acc = acc
                torch.save({'state_dict': model.state_dict(), 'best_accu': acc,}, path)
            if(acc>print_max_acc):
                print_max_acc = acc
            if(args.resume==True):
                f = open(file_name,'a')
                f.write(str(acc))
                f.write('\n')
        if(args.resume==True):
            f = open(file_name,'a')
            f.write('The max accu of the {} runs is:'.format(i))
            f.write(str(print_max_acc))
            f.write('\n')
    # pdb.set_trace()
    accu = torch.tensor(accu)
    accu = accu.view(args.folds,args.max_epoch)
    _, argmax = accu.max(dim=1)
    #accu = accu[torch.arange(args.folds, dtype=torch.long), argmax]
    acc_mean = accu.mean().item()
    acc_std = accu.std().item()
    desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
    if(args.resume==True):
        f = open(file_name,'a')
        f.write('The result is:')
        f.write(desc)
        f.write('\n')
print("finish")

C:\Users\Dell\AppData\Local\Programs\Python\Python310\lib\site-packages\torch_geometric\data\in_memory_dataset.py:183: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)

0it [02:46, ?it/s]


Initial x shape: torch.Size([4282, 3])
Initial edge_index shape: torch.Size([2, 15874])
Initial batch shape: torch.Size([4282])
Initial x shape: torch.Size([4766, 3])
Initial edge_index shape: torch.Size([2, 17304])
Initial batch shape: torch.Size([4766])
Initial x shape: torch.Size([4875, 3])
Initial edge_index shape: torch.Size([2, 18050])
Initial batch shape: torch.Size([4875])
Initial x shape: torch.Size([5697, 3])
Initial edge_index shape: torch.Size([2, 20968])
Initial batch shape: torch.Size([5697])
Initial x shape: torch.Size([5924, 3])
Initial edge_index shape: torch.Size([2, 22316])
Initial batch shape: torch.Size([5924])
Initial x shape: torch.Size([907, 3])
Initial edge_index shape: torch.Size([2, 3396])
Initial batch shape: torch.Size([907])
Initial x shape: torch.Size([6774, 3])
Initial edge_index shape: torch.Size([2, 25760])
Initial batch shape: torch.Size([6774])
Initial x shape: torch.Size([2039, 3])
Initial edge_index shape: torch.Size([2, 7494])
Initial batch shape:


0it [00:02, ?it/s, Train_Loss=0.670, Val_Loss=1.311, Acc=0.596, Epoch=000.0, Fold=000.0]

Initial x shape: torch.Size([2336, 3])
Initial edge_index shape: torch.Size([2, 8712])
Initial batch shape: torch.Size([2336])


0it [00:02, ?it/s, Train_Loss=0.670, Val_Loss=1.311, Acc=0.596, Epoch=000.0, Fold=000.0]

...

0it [00:01, ?it/s, Train_Loss=0.501, Val_Loss=3.532, Acc=0.405, Epoch=098.0, Fold=004.0]

Initial x shape: torch.Size([4370, 3])
Initial edge_index shape: torch.Size([2, 16230])
Initial batch shape: torch.Size([4370])


Initial x shape: torch.Size([4960, 3])
Initial edge_index shape: torch.Size([2, 18342])
Initial batch shape: torch.Size([4960])
Initial x shape: torch.Size([4743, 3])
Initial edge_index shape: torch.Size([2, 17890])
Initial batch shape: torch.Size([4743])
Initial x shape: torch.Size([4889, 3])
Initial edge_index shape: torch.Size([2, 18572])
Initial batch shape: torch.Size([4889])
Initial x shape: torch.Size([4975, 3])
Initial edge_index shape: torch.Size([2, 18754])
Initial batch shape: torch.Size([4975])
Initial x shape: torch.Size([1182, 3])
Initial edge_index shape: torch.Size([2, 4396])
Initial batch shape: torch.Size([1182])
Initial x shape: torch.Size([7230, 3])
Initial edge_index shape: torch.Size([2, 26566])
Initial batch shape: torch.Size([7230])
Initial x shape: torch.Size([2309, 3])
Initial edge_index shape: torch.Size([2, 8084])
Initial batch shape: torch.Size([2309])


0it [00:01, ?it/s, Train_Loss=0.493, Val_Loss=0.902, Acc=0.455, Epoch=099.0, Fold=004.0]

Initial x shape: torch.Size([6774, 3])
Initial edge_index shape: torch.Size([2, 25760])
Initial batch shape: torch.Size([6774])
Initial x shape: torch.Size([2039, 3])
Initial edge_index shape: torch.Size([2, 7494])
Initial batch shape: torch.Size([2039])
finish


## Manual Measurement

In [16]:

Eva_final=dict()

Base_model_accuracy=[]
T_base_model=[]
Num_parm_base_model=[]
Base_model_size=[]

In [17]:

for i in range(4):
    print('********************************************')
    print(f'The iteration is :{i+1} ')

    Eva=dict()
    accu = []
    accu_max = []
      # writer = SummaryWriter(log_dir=path2log)
    args.max_epoch=100
    max_acc = 0.79
    # model=qGIN(dataset, args.num_layers,hidden_units=args.hidden_units,bit=args.bit, quant_method=args.quant_method).to(device)
    # model = GIN(2,32).to(device)
    # setup_seed(12345)

    for fold, (train_idx, test_idx, val_idx) in enumerate(zip(*k_fold(dataset, args.folds))):
        print_max_acc=0
        train_dataset = dataset[train_idx.tolist()]
        test_dataset = dataset[test_idx.tolist()]
        val_dataset = dataset[val_idx.tolist()]
        train_loader = DataLoader(train_dataset, args.batch_size, shuffle=True,drop_last=False)
        val_loader = DataLoader(val_dataset, args.batch_size, shuffle=False,drop_last=False)
        test_loader = DataLoader(test_dataset, args.batch_size, shuffle=False,drop_last=False)
        k=0
        model=GIN(train_dataset, args.num_layers,hidden_units=args.hidden_units,
                num_deg=args.num_deg).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=args.lr_step_size, gamma=args.lr_decay_factor)


        # if (os.path.exists(path2check)):
        #     model = load_checkpoint(model,path2check)

        for epoch in range(args.max_epoch):
            t = tqdm(epoch)
            train_loss=0
            train_loss = train(model,optimizer,train_loader,args.a_loss, args.a_storage)
            start = time.process_time()
            val_loss = eval_loss(model,val_loader)
            end = time.process_time()
            acc = eval_acc(model,test_loader)
            # for name,param in model.named_parameters():
            #     a=param.grad
            #     if(a!=None):
            #         writer.add_histogram(tag=name+'_grad', values=a, global_step=epoch)
            scheduler.step()
            t.set_postfix(
                        {
                            "Train_Loss": "{:05.3f}".format(train_loss),
                            "Val_Loss": "{:05.3f}".format(val_loss),
                            "Acc": "{:05.3f}".format(acc),
                            "Epoch":"{:05.1f}".format(epoch),
                            "Fold":"{:05.1f}".format(fold),
                        }
                    )
            accu.append(acc)
            if(acc>max_acc):
                path = path2check+'/'+args.model+'_'+str(hidden_units)+'_on_'+dataset_name+'_'+str(bit)+'bit-'+str(max_epoch)+'.pth.tar'
                max_acc = acc
                torch.save({'state_dict': model.state_dict(), 'best_accu': acc,}, path)
            if(acc>print_max_acc):
                print_max_acc = acc
            if(args.resume==True):
                f = open(file_name,'a')
                f.write(str(acc))
                f.write('\n')
        if(args.resume==True):
            f = open(file_name,'a')
            f.write('The max accu of the {} runs is:'.format(i))
            f.write(str(print_max_acc))
            f.write('\n')
    # pdb.set_trace()
    accu = torch.tensor(accu)
    accu = accu.view(args.folds,args.max_epoch)
    _, argmax = accu.max(dim=1)
    #accu = accu[torch.arange(args.folds, dtype=torch.long), argmax]
    acc_mean = accu.mean().item()
    acc_std = accu.std().item()
    desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
    if(args.resume==True):
        f = open(file_name,'a')
        f.write('The result is:')
        f.write(desc)
        f.write('\n')
    print("finish")
    t0=time.time()
    base_model_accuracy=eval_acc(model,test_loader)
    t1=time.time()
    t_base_model=t1 - t0
    ###
    base_model_size = get_model_size(model,count_nonzero_only=True)
    num_parm_base_model=get_num_parameters(model, count_nonzero_only=True)
    ###

    print(f"base model has accuracy on test set={base_model_accuracy:.2f}%")
    print(f"base model has size={base_model_size/MiB:.2f} MiB")
    print(f"The time inference of base model is ={t_base_model}") 
    print(f"The number of parametrs of base model is:{num_parm_base_model}")
    #print(f"base model has size={base_model_size/MiB:.2f} MiB, "
         # f"which is {base_model_size/base_model_size:.2f} X smaller than "
          #f"the {base_model_size/MiB:.2f} MiB base model")

    #Update my Eva dictionary
    Eva.update({'base model accuracy': base_model_accuracy,
                'time inference of base model': t_base_model,
                'number parmameters of base model': num_parm_base_model,
                'size of base model': base_model_size})  
    
    Base_model_accuracy.append(Eva['base model accuracy'])
    T_base_model.append(Eva['time inference of base model'])
    Num_parm_base_model.append(int(Eva['number parmameters of base model']))
    Base_model_size.append(int(Eva['size of base model']))





C:\Users\Dell\AppData\Local\Programs\Python\Python310\lib\site-packages\torch_geometric\data\in_memory_dataset.py:183: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)


********************************************
The iteration is :1 


0it [00:12, ?it/s]
0it [00:01, ?it/s, Train_Loss=0.671, Val_Loss=0.829, Acc=0.507, Epoch=000.0, Fold=000.0]
0it [00:01, ?it/s, Train_Loss=0.671, Val_Loss=0.829, Acc=0.507, Epoch=000.0, Fold=000.0]

0it [00:01, ?it/s, Train_Loss=0.645, Val_Loss=0.952, Acc=0.646, Epoch=001.0, Fold=000.0]
0it [00:01, ?it/s, Train_Loss=0.613, Val_Loss=1.006, Acc=0.646, Epoch=002.0, Fold=000.0]
0it [00:01, ?it/s, Train_Loss=0.613, Val_Loss=1.006, Acc=0.646, Epoch=002.0, Fold=000.0]

0it [00:01, ?it/s, Train_Loss=0.633, Val_Loss=0.725, Acc=0.632, Epoch=003.0, Fold=000.0]
0it [00:01, ?it/s, Train_Loss=0.605, Val_Loss=0.944, Acc=0.632, Epoch=004.0, Fold=000.0]
0it [00:01, ?it/s, Train_Loss=0.605, Val_Loss=0.944, Acc=0.632, Epoch=004.0, Fold=000.0]

0it [00:01, ?it/s, Train_Loss=0.593, Val_Loss=0.623, Acc=0.587, Epoch=005.0, Fold=000.0]
0it [00:01, ?it/s, Train_Loss=0.571, Val_Loss=0.593, Acc=0.668, Epoch=006.0, Fold=000.0]
0it [00:01, ?it/s, Train_Loss=0.571, Val_Loss=0.593, Acc=0.668, Epoch=006.0, Fold=000.0]

0it [00:01, ?it/s, Train_Loss=0.467, Val_Loss=2.258, Acc=0.399, Epoch=060.0, Fold=000.0]
0it [00:01, ?it/s, Train_Loss=0.467, Val_Loss=2.258, Acc=0.399, Epoch=060.0, Fold=000.0]

0it [00:01, ?it/s, Train_Loss=0.443, Val_Loss=1.351, Acc=0.422, Epoch=061.0, Fold=000.0]
0it [00:01, ?it/s, Train_Loss=0.453, Val_Loss=0.562, Acc=0.691, Epoch=062.0, Fold=000.0]
0it [00:01, ?it/s, Train_Loss=0.453, Val_Loss=0.562, Acc=0.691, Epoch=062.0, Fold=000.0]

0it [00:01, ?it/s, Train_Loss=0.458, Val_Loss=0.833, Acc=0.498, Epoch=063.0, Fold=000.0]
0it [00:01, ?it/s, Train_Loss=0.443, Val_Loss=0.552, Acc=0.628, Epoch=064.0, Fold=000.0]
0it [00:01, ?it/s, Train_Loss=0.443, Val_Loss=0.552, Acc=0.628, Epoch=064.0, Fold=000.0]

0it [00:01, ?it/s, Train_Loss=0.428, Val_Loss=0.730, Acc=0.619, Epoch=065.0, Fold=000.0]
0it [00:01, ?it/s, Train_Loss=0.448, Val_Loss=1.109, Acc=0.466, Epoch=066.0, Fold=000.0]
0it [00:01, ?it/s, Train_Loss=0.448, Val_Loss=1.109, Acc=0.466, Epoch=066.0, Fold=000.0]

0it [00:01, ?it/s

0it [00:01, ?it/s, Train_Loss=0.529, Val_Loss=4.477, Acc=0.404, Epoch=020.0, Fold=001.0]
0it [00:01, ?it/s, Train_Loss=0.529, Val_Loss=4.477, Acc=0.404, Epoch=020.0, Fold=001.0]

0it [00:01, ?it/s, Train_Loss=0.513, Val_Loss=4.613, Acc=0.404, Epoch=021.0, Fold=001.0]
0it [00:01, ?it/s, Train_Loss=0.542, Val_Loss=3.762, Acc=0.404, Epoch=022.0, Fold=001.0]
0it [00:01, ?it/s, Train_Loss=0.542, Val_Loss=3.762, Acc=0.404, Epoch=022.0, Fold=001.0]

0it [00:01, ?it/s, Train_Loss=0.554, Val_Loss=99.007, Acc=0.404, Epoch=023.0, Fold=001.0]
0it [00:01, ?it/s, Train_Loss=0.560, Val_Loss=5.852, Acc=0.404, Epoch=024.0, Fold=001.0]
0it [00:01, ?it/s, Train_Loss=0.560, Val_Loss=5.852, Acc=0.404, Epoch=024.0, Fold=001.0]

0it [00:01, ?it/s, Train_Loss=0.558, Val_Loss=4.474, Acc=0.404, Epoch=025.0, Fold=001.0]
0it [00:01, ?it/s, Train_Loss=0.527, Val_Loss=1.723, Acc=0.404, Epoch=026.0, Fold=001.0]
0it [00:01, ?it/s, Train_Loss=0.527, Val_Loss=1.723, Acc=0.404, Epoch=026.0, Fold=001.0]

0it [00:01, ?it/

...

0it [00:01, ?it/s, Train_Loss=0.452, Val_Loss=7.720, Acc=0.404, Epoch=099.0, Fold=002.0]
0it [00:01, ?it/s, Train_Loss=0.672, Val_Loss=1.013, Acc=0.595, Epoch=000.0, Fold=003.0]
0it [00:01, ?it/s, Train_Loss=0.672, Val_Loss=1.013, Acc=0.595, Epoch=000.0, Fold=003.0]

0it [00:01, ?it/s, Train_Loss=0.647, Val_Loss=2.141, Acc=0.640, Epoch=001.0, Fold=003.0]
0it [00:01, ?it/s, Train_Loss=0.627, Val_Loss=1.224, Acc=0.658, Epoch=002.0, Fold=003.0]
0it [00:01, ?it/s, Train_Loss=0.627, Val_Loss=1.224, Acc=0.658, Epoch=002.0, Fold=003.0]

0it [00:01, ?it/s, Train_Loss=0.627, Val_Loss=0.901, Acc=0.631, Epoch=003.0, Fold=003.0]
0it [00:01, ?it/s, Train_Loss=0.605, Val_Loss=0.798, Acc=0.698, Epoch=004.0, Fold=003.0]
0it [00:01, ?it/s, Train_Loss=0.605, Val_Loss=0.798, Acc=0.698, Epoch=004.0, Fold=003.0]

0it [00:01, ?it/s, Train_Loss=0.603, Val_Loss=1.145, Acc=0.707, Epoch=005.0, Fold=003.0]
0it [00:01, ?it/s, Train_Loss=0.595, Val_Loss=1.115, Acc=0.671, Epoch=006.0, Fold=003.0]
0it [00:01, ?it/s,

0it [00:01, ?it/s, Train_Loss=0.484, Val_Loss=1.626, Acc=0.410, Epoch=059.0, Fold=003.0]
0it [00:01, ?it/s, Train_Loss=0.474, Val_Loss=3.736, Acc=0.437, Epoch=060.0, Fold=003.0]
0it [00:01, ?it/s, Train_Loss=0.474, Val_Loss=3.736, Acc=0.437, Epoch=060.0, Fold=003.0]

0it [00:01, ?it/s, Train_Loss=0.478, Val_Loss=34.441, Acc=0.473, Epoch=061.0, Fold=003.0]
0it [00:01, ?it/s, Train_Loss=0.466, Val_Loss=0.675, Acc=0.595, Epoch=062.0, Fold=003.0]
0it [00:01, ?it/s, Train_Loss=0.466, Val_Loss=0.675, Acc=0.595, Epoch=062.0, Fold=003.0]

0it [00:01, ?it/s, Train_Loss=0.468, Val_Loss=0.640, Acc=0.703, Epoch=063.0, Fold=003.0]
0it [00:01, ?it/s, Train_Loss=0.478, Val_Loss=1.270, Acc=0.459, Epoch=064.0, Fold=003.0]
0it [00:01, ?it/s, Train_Loss=0.478, Val_Loss=1.270, Acc=0.459, Epoch=064.0, Fold=003.0]

0it [00:01, ?it/s, Train_Loss=0.463, Val_Loss=2.904, Acc=0.514, Epoch=065.0, Fold=003.0]
0it [00:01, ?it/s, Train_Loss=0.475, Val_Loss=7.137, Acc=0.423, Epoch=066.0, Fold=003.0]
0it [00:01, ?it/s

0it [00:01, ?it/s, Train_Loss=0.569, Val_Loss=50.011, Acc=0.595, Epoch=019.0, Fold=004.0]
0it [00:01, ?it/s, Train_Loss=0.580, Val_Loss=2.793, Acc=0.604, Epoch=020.0, Fold=004.0]
0it [00:01, ?it/s, Train_Loss=0.580, Val_Loss=2.793, Acc=0.604, Epoch=020.0, Fold=004.0]

0it [00:01, ?it/s, Train_Loss=0.544, Val_Loss=1.099, Acc=0.437, Epoch=021.0, Fold=004.0]
0it [00:01, ?it/s, Train_Loss=0.558, Val_Loss=3.373, Acc=0.419, Epoch=022.0, Fold=004.0]
0it [00:01, ?it/s, Train_Loss=0.558, Val_Loss=3.373, Acc=0.419, Epoch=022.0, Fold=004.0]

0it [00:01, ?it/s, Train_Loss=0.543, Val_Loss=2.592, Acc=0.419, Epoch=023.0, Fold=004.0]
0it [00:01, ?it/s, Train_Loss=0.527, Val_Loss=1.966, Acc=0.405, Epoch=024.0, Fold=004.0]
0it [00:01, ?it/s, Train_Loss=0.527, Val_Loss=1.966, Acc=0.405, Epoch=024.0, Fold=004.0]

0it [00:01, ?it/s, Train_Loss=0.536, Val_Loss=10314.629, Acc=0.590, Epoch=025.0, Fold=004.0]
0it [00:01, ?it/s, Train_Loss=0.522, Val_Loss=1.948, Acc=0.405, Epoch=026.0, Fold=004.0]
0it [00:01, ?

0it [00:01, ?it/s, Train_Loss=0.473, Val_Loss=1.172, Acc=0.536, Epoch=079.0, Fold=004.0]
0it [00:01, ?it/s, Train_Loss=0.480, Val_Loss=0.713, Acc=0.698, Epoch=080.0, Fold=004.0]
0it [00:01, ?it/s, Train_Loss=0.480, Val_Loss=0.713, Acc=0.698, Epoch=080.0, Fold=004.0]

0it [00:01, ?it/s, Train_Loss=0.475, Val_Loss=0.697, Acc=0.703, Epoch=081.0, Fold=004.0]
0it [00:01, ?it/s, Train_Loss=0.483, Val_Loss=1.882, Acc=0.410, Epoch=082.0, Fold=004.0]
0it [00:01, ?it/s, Train_Loss=0.483, Val_Loss=1.882, Acc=0.410, Epoch=082.0, Fold=004.0]

0it [00:01, ?it/s, Train_Loss=0.476, Val_Loss=0.684, Acc=0.730, Epoch=083.0, Fold=004.0]
0it [00:01, ?it/s, Train_Loss=0.452, Val_Loss=1.616, Acc=0.595, Epoch=084.0, Fold=004.0]
0it [00:01, ?it/s, Train_Loss=0.452, Val_Loss=1.616, Acc=0.595, Epoch=084.0, Fold=004.0]

0it [00:01, ?it/s, Train_Loss=0.450, Val_Loss=1.615, Acc=0.595, Epoch=085.0, Fold=004.0]
0it [00:01, ?it/s, Train_Loss=0.450, Val_Loss=0.762, Acc=0.640, Epoch=086.0, Fold=004.0]
0it [00:01, ?it/s,

finish
base model has accuracy on test set=0.66%
base model has size=0.17 MiB
The time inference of base model is =0.1440277099609375
The number of parametrs of base model is:43271


In [19]:
Eva_final=dict()
base_model_accuracy_mean = stat.mean(Base_model_accuracy)
base_model_accuracy_std =  stat.stdev(Base_model_accuracy)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)

Eva_final.update({'base model accuracy':float(format(base_model_accuracy_mean, '.3f'))})
Eva_final.update({'Std of base model accuracy':float(format(base_model_accuracy_std, '.3f'))})
                 
t_base_model_mean =stat.mean(T_base_model)
t_base_model_std =stat.stdev(T_base_model)  
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'time inference of base model':float(format(t_base_model_mean, '.3f'))})
Eva_final.update({'Std of time inference of base model':float(format(t_base_model_std, '.3f'))})


num_parm_base_model_mean = stat.mean(Num_parm_base_model)
num_parm_base_model_std = stat.stdev(Num_parm_base_model)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'number parmameters of base model':num_parm_base_model_mean})
Eva_final.update({'Std of number parmameters of base model':num_parm_base_model_std})

base_model_size_mean = stat.mean(Base_model_size)
base_model_size_std = stat.stdev(Base_model_size)
#desc = "{:.3f} ± {:.3f}".format(acc_mean,acc_std)
Eva_final.update({'base_model_size':base_model_size_mean})
Eva_final.update({'Std of base_model_size':base_model_size_std})


print(f"All measurement about Quatization process ")   
Eva_final

All measurement about Quatization process 


{'base model accuracy': 0.508,
 'Std of base model accuracy': 0.124,
 'time inference of base model': 0.143,
 'Std of time inference of base model': 0.002,
 'number parmameters of base model': 43271,
 'Std of number parmameters of base model': 0.0,
 'base_model_size': 1384672,
 'Std of base_model_size': 0.0}

In [20]:
# Create a table
table_data = [Base_model_accuracy,T_base_model,Num_parm_base_model, Base_model_size]
            
headers=['1', '2', '3', '4']
# Print the table
tabulate(table_data, headers, tablefmt='fancy_grid')
# New column data
first_column_data =  ['Base_model_accuracy','T_base_model','Num_parm_base_model', 'Base_model_size']

# Add a custom index column
headers=['1', '2', '3','4']
table_with_index = tabulate(table_data, headers=['parameters'] + headers,
                            showindex=first_column_data, tablefmt="fancy_grid", numalign="center")

# Print the extended table
print(table_with_index)

╒═════════════════════╤═════════════╤═════════════╤═════════════╤═════════════╕
│ parameters          │      1      │      2      │      3      │      4      │
╞═════════════════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ Base_model_accuracy │  0.405405   │   0.40991   │  0.554054   │  0.662162   │
├─────────────────────┼─────────────┼─────────────┼─────────────┼─────────────┤
│ T_base_model        │  0.144089   │  0.143121   │  0.139982   │  0.144028   │
├─────────────────────┼─────────────┼─────────────┼─────────────┼─────────────┤
│ Num_parm_base_model │    43271    │    43271    │    43271    │    43271    │
├─────────────────────┼─────────────┼─────────────┼─────────────┼─────────────┤
│ Base_model_size     │ 1.38467e+06 │ 1.38467e+06 │ 1.38467e+06 │ 1.38467e+06 │
╘═════════════════════╧═════════════╧═════════════╧═════════════╧═════════════╛
